# 03.1 — Statistical & Intermittent Time Series Benchmark (Validation Protocol v1.0.21)

Đánh giá chuẩn hóa 11 mô hình thống kê, bao gồm:
- **Heuristic Baselines**: Naive (Persistence), SeasonalNaive (chu kỳ tuần $s=7$)
- **Classical Time Series**: ARIMA(3,0,3), SARIMA_EDA(3,0,3)(1,1,1)7, AutoARIMA, AutoETS
- **Intermittent Demand**: CrostonOptimized, CrostonSBA, TSB(0.2, 0.2), ADIDA ($N=2$), IMAPA ($K=2$)

### Quy tắc giao thức cốt lõi:
1. **Single-Fit Policy (Decision M2)**: Fit duy nhất 1 lần trên tập **Dev Train** ($T=6,329$ ngày: `2001-01-01` đến `2018-04-30`), đóng băng toàn bộ tham số mô hình.
2. **Causal Rolling Origins trên Validation**: Đánh giá rolling origin đa bước ($H=7$) trên đúng **724 Validation Origins** (`2018-05-01` đến `2020-04-23`).
3. **Zero Final Test Access Rule**: Tuyệt đối không nạp hay tiếp cận nhãn Test set (`2020-05-01` đến `2026-04-30`).
4. **4 Point Forecast Metrics**: MAE, MSE, RMSE, MASE (mẫu số chuẩn $\text{scale}_{\text{DevTrain}} = 3.7015424$ mm).

In [1]:
import sys
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsforecast.models import (
    Naive,
    SeasonalNaive,
    ARIMA,
    AutoARIMA,
    AutoETS,
)

from src.evaluation.protocols import (
    load_canonical_data,
    compute_benchmark_metrics,
    CANONICAL_DEV_TRAIN_MASE_SCALE,
    ExecutionStatus,
)
from src.evaluation.metrics import evaluate_point_forecast_metrics
from src.models.nixtla.wrappers import (
    StatsForecastWrapper,
    FrozenIntermittentWrapper,
)

# Nạp dữ liệu chuẩn hóa (chỉ lấy Dev Train và Validation, loại bỏ Test)
train_full, val_set, _ = load_canonical_data()
print(f"Dev Train samples: {len(train_full):,}, Validation samples: {len(val_set):,}")


## 2. Chuẩn bị Dữ liệu & Thiết lập 724 Validation Origins ($H=7$)

In [2]:
# Ghép nối chuỗi liên tục Dev Train + Validation
df_eval = pd.concat([train_full, val_set], ignore_index=True)
y_eval = df_eval['Lượng mưa'].astype(float).values
y_train = train_full['Lượng mưa'].astype(float).values

num_val_origins = 724
dev_train_len = len(train_full)  # 6329
origin_indices = [dev_train_len + i for i in range(num_val_origins)]
origin_dates = val_set['Ngày'].iloc[:num_val_origins].values

# Ground truth targets (724, 7)
y_val_true = np.array([y_eval[idx + 1 : idx + 8] for idx in origin_indices], dtype=np.float64)
assert y_val_true.shape == (num_val_origins, 7), f"Shape mismatch: {y_val_true.shape}"

print(f"Đã thiết lập {num_val_origins} Validation origins:")
print(f"  - Origin đầu tiên : {origin_dates[0]}")
print(f"  - Origin cuối cùng: {origin_dates[-1]}")
print(f"  - Shape y_val_true : {y_val_true.shape}, Lượng mưa TB: {y_val_true.mean():.4f} mm")


## 3. Khởi tạo & Fit 11 Mô hình Thống kê trên Dev Train (Single-Fit Policy M2)

In [3]:
models_config = [
    # Group 1: Heuristic Baselines
    {
        "name": "Naive",
        "group": "Baseline",
        "strategy": "Heuristic Baseline",
        "param_count": 0,
        "wrapper": StatsForecastWrapper(Naive(), alias="Naive"),
    },
    {
        "name": "SeasonalNaive",
        "group": "Baseline",
        "strategy": "Heuristic Baseline",
        "param_count": 0,
        "wrapper": StatsForecastWrapper(SeasonalNaive(season_length=7), alias="SeasonalNaive"),
    },
    # Group 2: Classical Time Series
    {
        "name": "ARIMA",
        "group": "Statistical",
        "strategy": "Classical Time Series",
        "param_count": 7,
        "wrapper": StatsForecastWrapper(ARIMA(order=(3, 0, 3)), alias="ARIMA"),
    },
    {
        "name": "SARIMA_EDA",
        "group": "Statistical",
        "strategy": "Classical Time Series",
        "param_count": 9,
        "wrapper": StatsForecastWrapper(
            ARIMA(order=(3, 0, 3), seasonal_order=(1, 1, 1), season_length=7),
            alias="SARIMA_EDA",
        ),
    },
    {
        "name": "AutoARIMA",
        "group": "Statistical",
        "strategy": "Classical Time Series",
        "param_count": 10,
        "wrapper": StatsForecastWrapper(AutoARIMA(season_length=7), alias="AutoARIMA"),
    },
    {
        "name": "AutoETS",
        "group": "Statistical",
        "strategy": "Classical Time Series",
        "param_count": 6,
        "wrapper": StatsForecastWrapper(AutoETS(season_length=7), alias="AutoETS"),
    },
    # Group 3: Intermittent Demand (FrozenIntermittentWrapper N=2, K=2 per Gate 4)
    {
        "name": "CrostonOptimized",
        "group": "Statistical",
        "strategy": "Intermittent Model",
        "param_count": 2,
        "wrapper": FrozenIntermittentWrapper("CrostonOptimized", alias="CrostonOptimized"),
    },
    {
        "name": "CrostonSBA",
        "group": "Statistical",
        "strategy": "Intermittent Model",
        "param_count": 2,
        "wrapper": FrozenIntermittentWrapper("CrostonSBA", alias="CrostonSBA"),
    },
    {
        "name": "TSB",
        "group": "Statistical",
        "strategy": "Intermittent Model",
        "param_count": 2,
        "wrapper": FrozenIntermittentWrapper("TSB", alpha_d=0.2, alpha_p=0.2, alias="TSB"),
    },
    {
        "name": "ADIDA",
        "group": "Statistical",
        "strategy": "Intermittent Model",
        "param_count": 2,
        "wrapper": FrozenIntermittentWrapper("ADIDA", aggregation_chunk_size=2, alias="ADIDA"),
    },
    {
        "name": "IMAPA",
        "group": "Statistical",
        "strategy": "Intermittent Model",
        "param_count": 2,
        "wrapper": FrozenIntermittentWrapper("IMAPA", max_aggregation_level=2, alias="IMAPA"),
    },
]

print(f"Đã cấu hình {len(models_config)} mô hình thống kê & ngắt quãng.")


## 4. Huấn luyện Single-Fit & Dự báo Causal Rolling-Origins trên Validation

In [4]:
forecasts_dict = {}
benchmark_rows = []
horizon_rows = []

for item in models_config:
    m_name = item["name"]
    wrapper = item["wrapper"]
    print(f">>> Đang huấn luyện & dự báo: {m_name} ({item['strategy']})...")

    # 4a. Fit 1 lần duy nhất trên Dev Train
    t0 = time.time()
    wrapper.fit(y_train)
    t_fit = time.time() - t0

    # 4b. Dự báo Causal Rolling-Origins
    t0 = time.time()
    preds = wrapper.predict_rolling(y_eval, origin_indices, h=7)
    preds = np.clip(preds, 0.0, None)
    t_infer = time.time() - t0

    forecasts_dict[m_name] = preds
    print(f"    Fit: {t_fit:.2f}s | Infer: {t_infer:.2f}s ({(t_infer/num_val_origins)*1000:.2f} ms/origin)")

    # 4c. Tính metrics tổng hợp
    row_df = compute_benchmark_metrics(
        y_true=y_val_true,
        y_pred=preds,
        model_name=m_name,
        group=item["group"],
        strategy=item["strategy"],
        training_time=t_fit,
        inference_time=t_infer,
        parameter_count=item["param_count"],
        status=ExecutionStatus.SUCCESS,
        rain_threshold=1.0,
        scale=CANONICAL_DEV_TRAIN_MASE_SCALE,
        horizon=7,
    )
    benchmark_rows.append(row_df)

    # 4d. Tính metrics theo từng horizon h=1..7
    detailed = evaluate_point_forecast_metrics(y_true=y_val_true, y_pred=preds, scale=CANONICAL_DEV_TRAIN_MASE_SCALE, horizon=7)
    h_rec = {"model": m_name, "MAE": detailed["MAE"], "RMSE": detailed["RMSE"], "MASE": detailed["MASE"]}
    for h in range(1, 8):
        h_rec[f"MAE@h{h}"] = detailed[f"mae_h{h}"]
        h_rec[f"RMSE@h{h}"] = detailed[f"rmse_h{h}"]
        h_rec[f"MASE@h{h}"] = detailed[f"mase_lag1_h{h}"]
    horizon_rows.append(h_rec)

print("\nHoàn thành toàn bộ 11 mô hình thống kê & ngắt quãng.")


## 5. Bảng Xếp hạng Validation & Lưu Artifacts Chuẩn hóa

In [5]:
df_results = pd.concat(benchmark_rows, ignore_index=True).sort_values(by="MAE", ascending=True).reset_index(drop=True)
df_horizons = pd.DataFrame(horizon_rows)

# Lưu kết quả vào thư mục data/processed/results
results_dir = ROOT_DIR / "data" / "processed" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

val_csv = results_dir / "statistical_val_results.csv"
val_npz = results_dir / "statistical_val_forecasts.npz"
h_csv = results_dir / "statistical_val_horizon_breakdown.csv"

df_results.to_csv(val_csv, index=False)
df_horizons.to_csv(h_csv, index=False)

payload = {"y_true": y_val_true, "origin_dates": origin_dates}
payload.update(forecasts_dict)
np.savez_compressed(val_npz, **payload)

# Tương thích ngược với các file kết quả cũ
df_results.to_csv(results_dir / "statistical_results.csv", index=False)
np.savez_compressed(results_dir / "statistical_forecasts.npz", **payload)

print(">> Đã lưu thành công các artifacts:")
print(f"  - Results CSV: {val_csv}")
print(f"  - Forecasts NPZ: {val_npz}")
print(f"  - Horizon CSV: {h_csv}\n")

print("=== BẢNG XẾP HẠNG VALIDATION (11 Statistical Models, Ranked by MAE) ===")
display_cols = ['model', 'group', 'strategy', 'MAE', 'MSE', 'RMSE', 'MASE', 'training_time', 'inference_time', 'status']
display(df_results[display_cols])

print("\n=== CHI TIẾT THEO HORIZON (MAE tại h=1..7) ===")
display(df_horizons[['model', 'MAE'] + [f'MAE@h{h}' for h in range(1, 8)]])
